In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "Polonia Bytom_Pogoń  Grodzisk Mazowiecki_4068759.csv", low_memory=False
)
print(df.shape)

(3892, 248)


In [3]:
df[["event_type_name"]].value_counts().head(10)

event_type_name
Pass               894
Carries            732
Ball Receipt*      720
Shot               628
Pressure           240
Ball Recovery      103
Duel                62
Clearance           52
Offensive Duel      49
Defensive Duel      49
Name: count, dtype: int64

In [4]:
shots = df[df["event_type_name"] == "Shot"].drop_duplicates(subset="id").copy()
shots["time_s"] = shots["minute"] * 60 + shots["second"]
shots = shots.sort_values(["time_s", "index"]).reset_index(drop=True)

In [5]:
shots_gist = shots[
    ["minute", "second", "team_name", "player_name", "statsbomb_xg", "outcome_name"]
].copy()
shots_gist

,minute,second,team_name,player_name,statsbomb_xg,outcome_name
0,4,13,Polonia Bytom,Benedik Mioč,0.058596,Off T
1,5,6,Polonia Bytom,Kamil Wojtyra,0.100382,Blocked
2,13,38,Polonia Bytom,Kamil Wojtyra,0.028285,Off T
3,17,48,Polonia Bytom,Mikolaj Labojko,0.042174,Saved
4,21,13,Pogoń Grodzisk Mazowiecki,Kacper Los,0.197502,Goal
5,25,4,Pogoń Grodzisk Mazowiecki,Igor Korczakowski,0.054080,Blocked
6,25,7,Pogoń Grodzisk Mazowiecki,Damian Jaroń,0.063239,Blocked
7,27,40,Polonia Bytom,Dominik Konieczny,0.043911,Saved
8,27,43,Polonia Bytom,Lucjan Zielinski,0.035480,Goal
9,29,34,Pogoń Grodzisk Mazowiecki,Igor Korczakowski,0.040728,Blocked


In [6]:
team1, team2 = "Polonia Bytom", "Pogoń Grodzisk Mazowiecki"
is_goal = shots_gist["outcome_name"].eq("Goal")
team1_score = (is_goal & shots_gist["team_name"].eq(team1)).cumsum().shift(fill_value=0)
team2_score = (is_goal & shots_gist["team_name"].eq(team2)).cumsum().shift(fill_value=0)

shots_gist["match_state"] = np.select(
    [team1_score.eq(team2_score), team1_score.gt(team2_score)],
    ["Draw", f"{team1} winning"],
    default=f"{team2} winning",
)

In [7]:
xg_pivot = (
    shots_gist.groupby(["team_name", "match_state"])["statsbomb_xg"]
    .sum()
    .unstack(fill_value=0)
)
xg_pivot

match_state,Draw,Pogoń Grodzisk Mazowiecki winning,Polonia Bytom winning
team_name,,,
Pogoń Grodzisk Mazowiecki,0.609015,0.117319,0.016594
Polonia Bytom,0.923124,0.079390,0.066237


In [8]:
assert round(xg_pivot.values.sum(), 12) == round(shots["statsbomb_xg"].sum(), 12)

In [9]:
summary = pd.DataFrame(
    {
        "xG": xg_pivot.stack(),
        "xG against": pd.concat(
            [xg_pivot.loc[team2], xg_pivot.loc[team1]], keys=[team1, team2]
        ),
    }
)
summary["xG diff"] = summary["xG"] - summary["xG against"]
summary.index.names = ["Team", "Match state"]
summary.reset_index()

,Team,Match state,xG,xG against,xG diff
0,Pogoń Grodzisk Mazowiecki,Draw,0.609015,0.923124,-0.314109
1,Pogoń Grodzisk Mazowiecki,Pogoń Grodzisk Mazowiecki winning,0.117319,0.079390,0.037929
2,Pogoń Grodzisk Mazowiecki,Polonia Bytom winning,0.016594,0.066237,-0.049642
3,Polonia Bytom,Draw,0.923124,0.609015,0.314109
4,Polonia Bytom,Pogoń Grodzisk Mazowiecki winning,0.079390,0.117319,-0.037929
5,Polonia Bytom,Polonia Bytom winning,0.066237,0.016594,0.049642
